# KANBoost vs. "Interpretable Clinical Classification with Kolmogorov-Arnold Networks"

Reproduces the exact data loading/preprocessing (`load_data()`, `random_state=0`, `n_patients=1000`, `test_split=0.2`) from the paper's official repo:
https://github.com/Patricia-A-Apellaniz/classification_with_kans

Paper: [arXiv:2509.16750](https://arxiv.org/abs/2509.16750) -- "Interpretable Clinical Classification with Kolmogorov-Arnold Networks".

Their **KAAM** (Kolmogorov-Arnold Additive Model) is `f(x) = alpha + sum_j g_j(x_j)` -- structurally identical to `KANBoostClassifier(gam=True)`, so this is an apples-to-apples architectural comparison, not just "another benchmark".

Datasets (all 6 from the paper): `heart`, `diabetes_h`, `diabetes_130`, `obesity`, `obesity_bin`, `breast_cancer`.

**Bonus, not in the original paper**: Brier score and log-loss (probability calibration) -- the paper reports Accuracy/ROC-AUC/F1/Precision/Recall only, no calibration metric. KANBoost's own benchmarks this session found a real, repeatable ranking-vs-calibration gap; worth checking here too.

In [ ]:
!pip install -q --upgrade kanboost

import kanboost, torch
print("kanboost:", kanboost.__version__)
print("cuda available:", torch.cuda.is_available())

## 1. Download the paper's exact preprocessed data files

`heart_data.csv` / `diabetes_h_data.csv` / `diabetes_130_data.csv` are large (~40-50MB each); `obesity.csv` is small. `breast_cancer` doesn't need a download -- it's loaded via `sklearn.datasets.load_breast_cancer()`, exactly like the paper's own `data.py`.

In [ ]:
import os
import urllib.request

DATA_DIR = "kaam_paper_data"
os.makedirs(DATA_DIR, exist_ok=True)

BASE_URL = "https://raw.githubusercontent.com/Patricia-A-Apellaniz/classification_with_kans/master/data"
FILES = ["heart_data.csv", "diabetes_h_data.csv", "diabetes_130_data.csv", "obesity.csv"]

for fname in FILES:
    dest = os.path.join(DATA_DIR, fname)
    if not os.path.exists(dest):
        print(f"Downloading {fname} ...")
        urllib.request.urlretrieve(f"{BASE_URL}/{fname}", dest)
    else:
        print(f"{fname} already present.")
print("Done.")

## 2. Exact replica of the paper's `load_data()` (src/data.py)

Same subsampling (`n=1000, random_state=0`), same `test_split=0.2, random_state=0`, same obesity categorical-to-numeric mapping, same z-score scaling of high-cardinality numeric columns. Only difference from the original: `args['data_folder']` points at `DATA_DIR` above instead of the repo's own `data/` folder.

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split


def scale_numerical_data(data):
    cols_to_scale = [col for col in data.columns if len(data[col].unique()) > 10]
    data_norm = data[cols_to_scale].values
    data_norm = (data_norm - data_norm.mean(axis=0)) / data_norm.std(axis=0)
    data.loc[:, cols_to_scale] = data_norm
    return data


def load_data(dataset_name, data_folder=DATA_DIR, test_split=0.2, n_patients=1000):
    """Verbatim port of the paper's src/data.py:load_data, only the data
    folder is parameterized instead of read from an args dict."""
    if dataset_name in ['heart', 'diabetes_h', 'diabetes_130']:
        data = pd.read_csv(os.path.join(data_folder, f"{dataset_name}_data.csv"))
        data = data.sample(n=n_patients, random_state=0).reset_index(drop=True)
        target_name = data.columns[-1]
        x, y = data.drop(columns=[target_name]), data[target_name]

    elif dataset_name == 'obesity' or dataset_name == 'obesity_bin':
        data = pd.read_csv(os.path.join(data_folder, 'obesity.csv'))
        data = data.sample(n=n_patients, random_state=0).reset_index(drop=True)

        data['Gender'] = data['Gender'].apply(lambda x: 1 if x == 'Female' else 0)
        data['family_history_with_overweight'] = data['family_history_with_overweight'].apply(lambda x: 1 if x == 'yes' else 0)
        data['FAVC'] = data['FAVC'].apply(lambda x: 1 if x == 'yes' else 0)
        data['CAEC'] = data['CAEC'].apply(lambda x: 3 if x == 'Always' else (2 if x == 'Frequently' else (1 if x == 'Sometimes' else 0)))
        data['SMOKE'] = data['SMOKE'].apply(lambda x: 1 if x == 'yes' else 0)
        data['SCC'] = data['SCC'].apply(lambda x: 1 if x == 'yes' else 0)
        data['CALC'] = data['CALC'].apply(lambda x: 3 if x == 'Always' else (2 if x == 'Frequently' else (1 if x == 'Sometimes' else 0)))
        data['MTRANS'] = data['MTRANS'].apply(lambda x: 4 if x == 'Automobile' else (3 if x == 'Motorbike' else (2 if x == 'Bike' else (1 if x == 'Public_Transportation' else 0))))
        data['NObeyesdad'] = data['NObeyesdad'].apply(lambda x: 6 if x == 'Obesity_Type_III' else (5 if x == 'Obesity_Type_II' else (4 if x == 'Obesity_Type_I' else (3 if x == 'Overweight_Level_II' else (2 if x == 'Overweight_Level_I' else (1 if x == 'Normal_Weight' else 0))))))

        if dataset_name == 'obesity_bin':
            data['NObeyesdad'] = data['NObeyesdad'].apply(lambda x: 1 if x > 3 else 0)

        data = data.fillna(data.mean())
        target_name = 'NObeyesdad'
        x, y = data.drop(columns=[target_name]), data[target_name]
        x = scale_numerical_data(x)

    elif dataset_name == 'breast_cancer':
        data = load_breast_cancer(as_frame=True)
        x, y = data.data, data.target
        x.columns = [col.replace(' ', '_') for col in x.columns]
        x = scale_numerical_data(x)

    else:
        raise ValueError(f"Data name {dataset_name} not found")

    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=test_split, random_state=0)
    return x_train, x_test, y_train, y_test


DATASETS = ['heart', 'diabetes_h', 'diabetes_130', 'obesity', 'obesity_bin', 'breast_cancer']

## 3. Train KANBoost on each dataset (exact same train/test split as the paper)

`gam=True` to match KAAM's architecture directly. Metrics: the paper's five (Accuracy, ROC-AUC macro-OVR, F1, Precision, Recall, all macro-averaged for multiclass) plus Brier score and log-loss (not in the paper).

In [ ]:
import time
import numpy as np
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score, precision_score, recall_score,
    brier_score_loss, log_loss,
)
from kanboost import KANBoostClassifier


def evaluate(y_true, y_pred, proba, n_classes):
    avg = 'binary' if n_classes == 2 else 'macro'
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, average=avg, zero_division=0),
        'precision': precision_score(y_true, y_pred, average=avg, zero_division=0),
        'recall': recall_score(y_true, y_pred, average=avg, zero_division=0),
    }
    try:
        if n_classes == 2:
            metrics['roc_auc'] = roc_auc_score(y_true, proba[:, 1])
            metrics['brier'] = brier_score_loss(y_true, proba[:, 1])
        else:
            metrics['roc_auc'] = roc_auc_score(y_true, proba, multi_class='ovr', average='macro')
            metrics['brier'] = np.nan  # brier_score_loss is binary-only in sklearn
        metrics['log_loss'] = log_loss(y_true, proba)
    except ValueError as e:
        metrics['roc_auc'] = metrics['brier'] = metrics['log_loss'] = np.nan
        print(f"  (metric warning: {e})")
    return metrics


results = []
for name in DATASETS:
    print(f"\n=== {name} ===")
    x_train, x_test, y_train, y_test = load_data(name)
    n_classes = y_train.nunique()
    print(f"  train={len(x_train)}  test={len(x_test)}  features={x_train.shape[1]}  classes={n_classes}")

    model = KANBoostClassifier(
        n_estimators=60, kan_hidden=1, kan_grid=3, kan_steps=15,
        gam=True, early_stopping_rounds=10, validation_fraction=0.15,
        device=None, random_state=0, verbose=False,
    )
    t0 = time.time()
    model.fit(x_train, y_train)
    fit_time = time.time() - t0

    t0 = time.time()
    proba = model.predict_proba(x_test)
    pred = model.predict(x_test)
    predict_time = time.time() - t0

    metrics = evaluate(y_test.values, pred, proba, n_classes)
    metrics.update({'dataset': name, 'fit_time_s': fit_time, 'predict_time_s': predict_time})
    results.append(metrics)
    print(f"  acc={metrics['accuracy']:.4f}  auc={metrics['roc_auc']:.4f}  "
          f"f1={metrics['f1']:.4f}  brier={metrics['brier']}  logloss={metrics['log_loss']:.4f}  "
          f"fit={fit_time:.1f}s")

results_df = pd.DataFrame(results)[
    ['dataset', 'accuracy', 'roc_auc', 'f1', 'precision', 'recall', 'brier', 'log_loss', 'fit_time_s', 'predict_time_s']
]
results_df

## 4. Compare against the paper's published numbers

Hand-transcribed from Table 2 of arXiv:2509.16750 (Accuracy / ROC-AUC only -- the paper doesn't report Brier/log-loss at all, which is the point of the bonus columns above).

In [ ]:
paper_results = pd.DataFrame([
    # dataset,        model,          accuracy, roc_auc
    ['heart',         'LR',           0.79, 0.89],
    ['heart',         'RF',           0.90, 0.90],
    ['heart',         'Logistic-KAN', 0.84, 0.91],
    ['heart',         'KAAM',         0.82, 0.90],
    ['diabetes_h',    'LR',           0.80, 0.70],
    ['diabetes_h',    'Logistic-KAN', 0.82, 0.68],
    ['diabetes_h',    'KAAM',         0.82, 0.68],
    ['diabetes_130',  'LR',           0.55, 0.53],
    ['diabetes_130',  'Logistic-KAN', 0.51, 0.49],
    ['diabetes_130',  'KAAM',         0.55, 0.53],
    ['obesity',       'RF',           0.91, 1.00],
    ['obesity',       'Logistic-KAN', 0.98, 1.00],
    ['obesity',       'KAAM',         0.95, 1.00],
    ['obesity_bin',   'LR',           1.00, 1.00],
    ['obesity_bin',   'Logistic-KAN', 1.00, 1.00],
    ['obesity_bin',   'KAAM',         1.00, 1.00],
    ['breast_cancer', 'LR',           0.96, 1.00],
    ['breast_cancer', 'Logistic-KAN', 0.97, 0.99],
    ['breast_cancer', 'KAAM',         0.96, 1.00],
], columns=['dataset', 'model', 'accuracy', 'roc_auc'])

kanboost_rows = results_df[['dataset', 'accuracy', 'roc_auc']].copy()
kanboost_rows.insert(1, 'model', 'KANBoost')

comparison = pd.concat([paper_results, kanboost_rows], ignore_index=True)
comparison = comparison.sort_values(['dataset', 'roc_auc'], ascending=[True, False])
comparison

## 5. Per-dataset ranking: where does KANBoost land against KAAM/Logistic-KAN/LR/RF?

In [ ]:
for name in DATASETS:
    sub = comparison[comparison['dataset'] == name].sort_values('roc_auc', ascending=False).reset_index(drop=True)
    rank = sub.index[sub['model'] == 'KANBoost'][0] + 1
    print(f"{name:15s} KANBoost rank by ROC-AUC: {rank} of {len(sub)}")
    display(sub)